# Rate-KL replication: 3 datasets x 3 seeds + bimodality diagnostic

Single-seed Fashion-MNIST found rate-KL (rho=0.09) at lambda=0.01 beating the matched-sparsity TopK reference on steering impact (2.4x), mean purity (0.70 vs 0.28), and top-10%-by-importance purity (0.49 vs 0.035 -- not a hub artifact), at a 2.7x MSE cost; lambda=0.001 matched TopK's steering with ~2x purity at 1.6x MSE. This runs the same replication protocol that validated (then correctly complicated) the Gini result: fashion_mnist / mnist / cifar10, seeds 0-2, mean +/- std.

Each run also records the per-sample active-count distribution (quantiles, fraction of samples with <5 active features) to diagnose the `n_interventions` drop seen at high lambda -- rate-KL pins the *mean* active count but not per-sample counts, so activity can go bimodal, leaving some inputs near-silent.

**Win condition:** the lambda=0.01 pattern (steering > TopK, top10purity >> TopK, sparsity matched) holds with non-overlapping error bars on all three datasets, and `lt5active` stays near 0 at the chosen operating points.

**Runtime:** 6 model-trainings per (dataset, seed) pair... 2 lambdas + TopK reference = 3 models x 9 pairs = 27 trainings. Expect roughly 2-3x the previous replication sweep's runtime on the same GPU.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

In [ ]:
# Two operating points: lambda=0.001 (TopK-steering-parity, higher purity,
# small MSE cost) and lambda=0.01 (2.4x steering, larger MSE cost). A TopK
# reference (k=round(rho*1024)=92) is trained per (dataset, seed).
!python run_rate_kl_replication.py --datasets fashion_mnist mnist cifar10 --seeds 0 1 2 --rho 0.09 --lambdas 0.001 0.01

In [ ]:
import json
with open('results/rate_kl/summary.json') as f:
    summary = json.load(f)

for dataset, models in summary.items():
    print(f"\n-- {dataset} --")
    for model, m in models.items():
        def fmt(key, prec=4):
            v = m.get(key)
            return f"{v['mean']:.{prec}f}+/-{v['std']:.{prec}f}" if v else 'n/a'
        print(f"{model:24s} mse={fmt('mse')}  sparsity={fmt('relative_sparsity', 3)}  "
              f"purity={fmt('mean_purity', 3)}  top10purity={fmt('mean_purity_top10pct_by_importance', 3)}  "
              f"ablate={fmt('steering_impact_ablate')}  clamp={fmt('steering_impact_clamp')}  "
              f"lt5active={fmt('frac_samples_lt5_active', 3)}")

In [ ]:
!zip -r rate_kl_replication_results.zip results/rate_kl
from google.colab import files
files.download('rate_kl_replication_results.zip')